In [ ]:
import os
import re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

# ----------------------------
# Config
# ----------------------------
MODEL = "gpt-5.4"   # 改成你实际用的最新模型名
REASONING_EFFORT_STEP1 = "medium"
REASONING_EFFORT_STEP2 = "medium"

BASE_DIR = Path("../")
PROMPTS_DIR = BASE_DIR / "prompts"
INPUTS_DIR = BASE_DIR / "inputs"
OUTPUTS_DIR = BASE_DIR / "outputs" / "drafts_v4" 

INPUT_PY = INPUTS_DIR / "HyyAnalysis.py"
PROMPT_REF = PROMPTS_DIR / "prompt_extract_reference.txt"
PROMPT_PUB_RUBRIC = PROMPTS_DIR / "prompt_generate_public_rubric.txt"
PROMPTS_LEVEL_RUBRIC = PROMPTS_DIR.rglob("prompt_generate_l*.txt")
PROMPTS_LEVEL_RUBRIC = sorted(PROMPTS_LEVEL_RUBRIC)

OUT_REF = OUTPUTS_DIR / "reference_analysis.md"
OUT_PUB_RUBRIC = OUTPUTS_DIR / "public_rubric.yaml"

load_dotenv(find_dotenv())
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def split_generated_files(text: str, output_dir: Path):
    pattern = r"=== FILE: (.+?) ===\n(.*?)(?=\n=== FILE: |\Z)"
    matches = re.findall(pattern, text, flags=re.S)

    output_dir.mkdir(parents=True, exist_ok=True)

    for filename, content in matches:
        path = output_dir / filename.strip()
        path.write_text(content.strip() + "\n", encoding="utf-8")
        print(f"Wrote {path}")

def call_model(prompt: str, effort: str) -> str:
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        reasoning={"effort": effort},
        max_output_tokens=10000,
    )
    return response.output_text.strip()


In [ ]:
py_text = read_text(INPUT_PY)
prompt_ref_template = read_text(PROMPT_REF)

prompt_ref = prompt_ref_template.replace("{{PYTHON_SCRIPT}}", py_text)
reference_md = call_model(prompt_ref, REASONING_EFFORT_STEP1)
write_text(OUT_REF, reference_md + "\n")
print(f"Wrote {OUT_REF}")

In [ ]:
prompt_pub_rubric_template = read_text(PROMPT_PUB_RUBRIC)
prompt_pub_rubric = prompt_pub_rubric_template.replace("{{REFERENCE_ANALYSIS_MD}}", reference_md)

pub_rubric_md = call_model(prompt_pub_rubric, REASONING_EFFORT_STEP2)
write_text(OUT_PUB_RUBRIC, pub_rubric_md + "\n")
print(f"Wrote {OUT_PUB_RUBRIC}")

In [2]:
PROMPTS_LEVEL_RUBRIC

[PosixPath('../prompts/prompt_generate_l1.txt'),
 PosixPath('../prompts/prompt_generate_l2.txt'),
 PosixPath('../prompts/prompt_generate_l3.txt')]

In [11]:
reference_md = read_text(OUT_REF)
for PROMPT_LEVEL_RUBRIC in PROMPTS_LEVEL_RUBRIC:
    prompt_level_rubric_template = read_text(PROMPT_LEVEL_RUBRIC)
    prompt_level_rubric = prompt_level_rubric_template.replace("{{REFERENCE_ANALYSIS_MD}}", reference_md)

    level_rubric_md = call_model(prompt_level_rubric, REASONING_EFFORT_STEP2)
    split_generated_files(level_rubric_md, OUTPUTS_DIR)

Wrote ../outputs/drafts_v3/l1_prompt.md
Wrote ../outputs/drafts_v3/private_rubric_l1.yaml
Wrote ../outputs/drafts_v3/l2_prompt.md
Wrote ../outputs/drafts_v3/private_rubric_l2.yaml
Wrote ../outputs/drafts_v3/l3_prompt.md
Wrote ../outputs/drafts_v3/private_rubric_l3.yaml


In [9]:
level_rubric_md

''